# Generate Elevator Risk Predictions

Re-trains the best model from Module 3 (RandomForestClassifier) on `data/feature_matrix.csv` and generates a risk score for every unique elevator in the fleet.

**Run from project root** (same directory as `data/` and `intelligence/`).

Output: `data/predictions.csv` — loaded by the Go API at startup to serve `/api/elevators/{id}/risk`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from datetime import date

# ── Configuration ─────────────────────────────────────────────────────────────
FEATURES = [
    "prior_inspection_count", "prior_fail_count", "prior_pass_count",
    "prior_pass_rate", "days_since_last_inspection", "prior_inspection_frequency",
    "last_inspection_outcome", "prior_order_count", "max_prior_riskscore",
    "mean_prior_riskscore", "prior_overdue_order_count", "prior_unresolved_order_count",
    "distinct_directive_count", "device_type_encoded",
    "insp_type_Followup", "insp_type_Other", "insp_type_Periodic",
]
MODEL_VERSION = "v4.1"
SPLIT_DATE    = "2015-12-14"  # temporal split used in ml_pipeline.ipynb

def assign_risk_level(score: float) -> str:
    if score >= 0.7:
        return "HIGH"
    if score >= 0.4:
        return "MEDIUM"
    return "LOW"

In [ ]:
# ── 1. Load feature matrix ────────────────────────────────────────────────────
df = pd.read_csv("data/feature_matrix.csv")
df["Latest_INSPECTION_Date"] = pd.to_datetime(df["Latest_INSPECTION_Date"])

print(f"Feature matrix loaded : {len(df):,} rows")
print(f"Unique elevators      : {df['ElevatingDevicesNumber'].nunique():,}")

In [ ]:
# ── 2. Train model (temporal split — identical to ml_pipeline.ipynb) ──────────
train = df[df["Latest_INSPECTION_Date"] < SPLIT_DATE]
X_train = train[FEATURES]
y_train = train["target"]

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=100, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )),
])
pipeline.fit(X_train, y_train)
print(f"Trained on {len(train):,} rows (before {SPLIT_DATE})")

In [ ]:
# ── 3. Generate predictions (latest row per elevator = current feature state) ─
latest = (
    df.sort_values("Latest_INSPECTION_Date")
      .groupby("ElevatingDevicesNumber", sort=False)
      .last()
      .reset_index()
)

proba = pipeline.predict_proba(latest[FEATURES])[:, 1]  # P(NOT PASS)

today = date.today().isoformat()
predictions = pd.DataFrame({
    "elevator_id":    latest["ElevatingDevicesNumber"].astype(str),
    "risk_score":     proba.round(4),
    "risk_level":     [assign_risk_level(s) for s in proba],
    "model_version":  MODEL_VERSION,
    "prediction_date": today,
})

In [ ]:
# ── 4. Validate output ────────────────────────────────────────────────────────

# All unique elevators from the feature matrix must be represented
fm_ids   = set(df["ElevatingDevicesNumber"].astype(str).unique())
pred_ids = set(predictions["elevator_id"].unique())
missing  = fm_ids - pred_ids
assert len(missing) == 0, f"Missing predictions for {len(missing)} elevator(s): {list(missing)[:5]}"

# All risk scores must be within [0, 1]
assert predictions["risk_score"].between(0, 1).all(), "risk_score out of [0, 1]"

# Distribution must not be degenerate (no single category > 95%)
dist = predictions["risk_level"].value_counts(normalize=True)
assert dist.max() < 0.95, f"Degenerate distribution: {dist.to_dict()}"

print("✅ All validations passed")

In [ ]:
# ── 5. Summary ────────────────────────────────────────────────────────────────
counts = predictions["risk_level"].value_counts()

print(f"Total elevators       : {len(predictions):,}")
print(f"\nRisk level counts     :")
for level in ["HIGH", "MEDIUM", "LOW"]:
    n = counts.get(level, 0)
    pct = n / len(predictions) * 100
    print(f"  {level:<8}: {n:>6,}  ({pct:.1f}%)")
print(f"\nRisk score statistics :")
print(f"  Min  : {predictions['risk_score'].min():.4f}")
print(f"  Max  : {predictions['risk_score'].max():.4f}")
print(f"  Mean : {predictions['risk_score'].mean():.4f}")

In [ ]:
# ── 6. Save ───────────────────────────────────────────────────────────────────
# Enforce column order explicitly — Go API reads predictions.csv positionally.
COL_ORDER = ["elevator_id", "risk_score", "risk_level", "model_version", "prediction_date"]
predictions[COL_ORDER].to_csv("data/predictions.csv", index=False)
print(f"Saved → data/predictions.csv  ({len(predictions):,} rows, model {MODEL_VERSION})")